In [2]:
%pip install pandas geopandas plotly scikit-learn numpy
import pandas as pd
import geopandas as gpd
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import numpy as np
import sqlite3


[notice] A new release of pip available: 22.2.2 -> 25.0
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
def load_and_transform_energy_data():
    # Read CSV with proper data types
    df = pd.read_csv('energy-and-utilities-linc.csv',
                     delimiter=';',
                     names=['County', 'Empty', 'Year', 'Variable', 'Value'],
                     dtype={'County': str, 'Empty': str, 'Year': str, 'Variable': str, 'Value': str})
    
    # Remove header rows and clean data
    df = df[~df['County'].isin(['Area Name', 'County'])]
    df = df[df['Value'].str.isnumeric().fillna(False)]

    # Drop empty column and convert types
    df = df.drop('Empty', axis=1)
    df['Value'] = pd.to_numeric(df['Value'])
    df['Year'] = pd.to_numeric(df['Year'])
    
    # Create pivot table
    transformed_df = pd.pivot_table(
        df,
        values='Value',
        index=['County', 'Year'],
        columns='Variable',
        aggfunc='first'
    ).reset_index()
    
    # Rename columns for clarity
    column_mapping = {
        'Occupied Housing Units Heated by Electricity': 'heated_by_electricity',
        'Occ Housing Units Heated by Gas Piped Underground': 'heated_by_gas',
        'Occupied Housing Units Heated by Fuel Oil': 'heated_by_fuel_oil',
        'Occ Housing Units Heated by Coal, Wood, Solar, or Other': 'heated_by_other',
        'Occupied Housing Units without House Heating Fuel': 'no_heating',
        'Occ Housing Units Heated by Bottled Tank or LP Gas Fuel': 'heated_by_lp_gas'
    }
    
    # Convert types and fill nulls
    transformed_df = transformed_df.rename(columns=column_mapping)
    transformed_df = transformed_df.fillna(0)
    
     # Convert to integers after cleaning
    numeric_cols = transformed_df.columns.difference(['County'])
    transformed_df[numeric_cols] = transformed_df[numeric_cols].astype(int)
    

    return transformed_df

# Cell 4: Create SQLite Database
def create_database():
    # Get transformed data
    energy_df = load_and_transform_energy_data()
    
    # Create database connection
    conn = sqlite3.connect('nc_energy.db')
    
    # Create table schema
    create_table_sql = '''
    CREATE TABLE IF NOT EXISTS energy_consumption (
        County TEXT,
        Year INTEGER,
        heated_by_electricity INTEGER,
        heated_by_gas INTEGER,
        heated_by_fuel_oil INTEGER,
        heated_by_other INTEGER,
        no_heating INTEGER,
        heated_by_lp_gas INTEGER,
        PRIMARY KEY (County, Year)
    ) WITHOUT ROWID;
    '''
    conn.executescript(create_table_sql)
    energy_df.to_sql('energy_consumption', conn, if_exists='replace', index=False)
    # Insert data
    energy_df.to_sql('energy_consumption', 
                     conn, 
                     if_exists='replace', 
                     index=False,
                     dtype={
                         'County': 'TEXT',
                         'Year': 'INTEGER',
                         'heated_by_electricity': 'INTEGER',
                         'heated_by_gas': 'INTEGER',
                         'heated_by_fuel_oil': 'INTEGER',
                         'heated_by_other': 'INTEGER',
                         'no_heating': 'INTEGER',
                         'heated_by_lp_gas': 'INTEGER'
                     })
    
    return conn, energy_df

def get_county_data(conn, county_name):
    query = '''
    SELECT 
        County,
        Year,
        heated_by_electricity,
        heated_by_gas,
        heated_by_fuel_oil,
        heated_by_other,
        no_heating,
        heated_by_lp_gas
    FROM energy_consumption 
    WHERE County = ?
    ORDER BY Year;
    '''
    return pd.read_sql(query, conn, params=(county_name,))

# Create database with new structure
energy_df = load_and_transform_energy_data()
print(energy_df.head())
print("\nColumns:", energy_df.columns.tolist())
print("\nData types:", energy_df.dtypes)

#Test get_county_data
conn, energy_df = create_database()
county_data = get_county_data(conn, 'Guilford County')
print(county_data.head())


Variable           County  Year  heated_by_lp_gas  heated_by_other  \
0         Alamance County  1990              4006             2541   
1         Alamance County  2000              6511             1005   
2         Alamance County  2010              6194             1243   
3         Alamance County  2015              5119             1918   
4         Alamance County  2020              4005             1101   

Variable  heated_by_gas  heated_by_electricity  heated_by_fuel_oil  no_heating  
0                 15946                  12543                7552          64  
1                 24211                  16783                2996          78  
2                 26390                  23032                2086          55  
3                 26122                  26576                1578         232  
4                 26842                  32290                 848         369  

Columns: ['County', 'Year', 'heated_by_lp_gas', 'heated_by_other', 'heated_by_gas', 'heated_

In [4]:
# Cell 3: Load Geographic Data
def load_geo_data():
    """Load and process geographic data"""
    counties_gdf = pd.read_csv('NCCountyCoordinates.csv')
    return counties_gdf
    
counties_gdf = load_geo_data()

In [14]:
# Cell 5: Query Functions
def get_county_data(conn, county_name):
    query = '''
    SELECT * FROM energy_consumption 
    WHERE county = ? 
    ORDER BY Year
    '''
    return pd.read_sql_query(query, conn, params=[county_name])

def get_county_list(conn):
    query = '''
    SELECT DISTINCT county FROM energy_consumption
    '''
    return pd.read_sql_query(query, conn)

#Print the returned queries
print(get_county_list(conn))
print(get_county_data(conn, 'Guilford County').head())

              County
0    Alamance County
1   Alexander County
2   Alleghany County
3       Anson County
4        Ashe County
..               ...
95      Wayne County
96     Wilkes County
97     Wilson County
98     Yadkin County
99     Yancey County

[100 rows x 1 columns]
            County  Year  heated_by_lp_gas  heated_by_other  heated_by_gas  \
0  Guilford County  1990              4291             5438          48770   
1  Guilford County  2000              7941             2034          76608   
2  Guilford County  2010              7944             1771          85699   
3  Guilford County  2015              6445             1999          82119   
4  Guilford County  2020              6132             1926          85206   

   heated_by_electricity  heated_by_fuel_oil  no_heating  
0                  57535               21482         190  
1                  71024               10715         345  
2                  87088                6580         479  
3                 1

In [ ]:
# Cell 6: Model Training
def train_prediction_model(df, target_variable='Value'):
    X = df[['Year']].values
    y = df[target_variable].values
    
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X, y)
    return model

# Cell 7: Prediction Function
def predict_energy_consumption(model, year):
    X_pred = np.array([year]).reshape(-1, 1)
    return model.predict(X_pred)

# Cell 8: Interactive Plot

def plot_energy_consumption(county_name, energy_df, counties_gdf, model):
    county_data = get_county_data(conn, county_name)
    
    fig = px.line(county_data, x='Year', y='Value', title=f'Energy Consumption for {county_name}')
    
    # Add predicted values
    years = pd.date_range(start='2022', end='2030', freq='Y')
    predictions = [predict_energy_consumption(model, year.year)[0] for year in years]
    predictions_df = pd.DataFrame({'Year': years, 'Value': predictions})
    
    fig.add_scatter(x=predictions_df['Year'], y=predictions_df['Value'], mode='lines', name='Predicted')
    
    # Add map
    county_geo = counties_gdf[counties_gdf['County'] == county_name]
    fig.update_geos(fitbounds='locations', visible=False)
    fig.add_trace(px.choropleth_mapbox(county_geo, geojson=county_geo.geometry, locations=county_geo.index, color='County', mapbox_style='carto-positron').data[0])
    
    fig.show()


In [ ]:
# Cell 7: Generate Predictions
def generate_predictions(model, start_year=2025, end_year=2030, step=5):
    future_years = range(start_year, end_year + 1, step)
    predictions = {}
    
    for year in future_years:
        pred = model.predict([[year]])
        predictions[year] = pred[0]
        
    return pd.DataFrame(predictions.items(), columns=['Year', 'Predicted_Value'])


In [5]:
# Cell 8: Visualization
def create_choropleth_map(energy_df, counties_gdf, predictions=None):
    """Create interactive choropleth map"""
    # Combine historical and predicted data if available
    if predictions is not None:
        energy_df = pd.concat([energy_df, predictions])
    
    fig = px.choropleth(
        energy_df,
        geojson=counties_gdf,
        locations='Area_Name',
        featureidkey='properties.NAME',
        color='Value',
        animation_frame='Year',
        range_color=(0, energy_df['Value'].max()),
        scope="usa",
        title='NC Energy Consumption by County (1990-2030)',
        labels={'Value': 'Energy Consumption'}
    )
    
    fig.update_geos(
        fitbounds="locations",
        visible=False,
        center={"lat": 35.5, "lon": -80},
        scope='usa',
    )
    
    return fig